**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Statistical Signal Processing

Real signals are random. This workshop supplies the theory the [adaptive filtering](../Intro_Time_Series/README.md) notebooks borrowed on credit: random processes and stationarity, spectral estimation done honestly, the Wiener filter derived, and matched filters & detection — the statistics of pulling signals out of noise.

## 1. Pre-requisites

- [Random Variables](../Intro_Math/Analysis/Random_Variables.ipynb) & [Independence](../Intro_Math/Analysis/Independence.ipynb).
- [Foundations of Signal Processing 1](./Foundations_of_Signal_Processing_1.ipynb) (DFT, convolution).
- [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) helps for Session 4.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *Random Processes & Stationarity* (~35 min)
**Goal:** treat a signal as a family of random variables; define autocorrelation and WSS.
**Builds on:** [Random Variables](../Intro_Math/Analysis/Random_Variables.ipynb). &nbsp; **Feeds into:** Session 2 (spectral estimation).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Random Processes & Stationarity</b></summary>

**Timing (~35 min).** 8 min the ensemble picture · 10 min autocorrelation · 10 min the demo · 7 min ergodicity, which is the load-bearing assumption.

**Board first — draw the ensemble, not the signal.** A random process is a random variable *per time index*: one run of the experiment produces an entire waveform. Draw five stacked realisations of the same process and circle a single time index — the values down that column are samples of one random variable. Students arrive thinking of a random signal as "a signal with noise on it"; the ensemble picture is what makes expectations like $E[x[n]x[m]]$ meaningful.

**Then the reduction that makes it tractable.** The full joint distribution is hopeless, so we keep two numbers: the mean and the autocorrelation $r[n,m] = E[x[n]x[m]]$ — literally "how much does this process remember itself across time." **WSS** says neither depends on absolute time, so $r$ collapses to a function of lag alone. Say plainly that WSS is an *assumption we impose to make progress*, not a property most real signals have; speech, music, and radar returns are all non-stationary, which is exactly why [Time–Frequency](./Time_Frequency_2.ipynb) and [Cyclostationarity](./Cyclostationary_HOS.ipynb) exist.

**Ergodicity deserves its own minute, because everything downstream rests on it.** We have *one* recording, not 400. Every practical estimate replaces an ensemble average with a time average, and that swap is legal only under ergodicity — the [LLN](../Intro_Math/Analysis/Independence.ipynb) applied along time rather than across trials. Point out that this cell deliberately uses `R = 400` realisations to compute a *true* ensemble average, which is a luxury no real experiment has. It is here so students can see what the time-average version is approximating.

**Ask the room before the plot.** "Two AR(1) processes, $a = +0.9$ and $a = -0.9$. Same magnitude of memory. How do they differ?" The positive one is smooth — each sample resembles the last. The negative one alternates — each sample is the *opposite* of the last, so it looks jittery while being just as strongly correlated. The point: correlation strength and visual smoothness are different things, and the sign of the memory decides which you get.

**Then connect it forward.** The positive-memory process has its energy at low frequency; the alternating one at high frequency. So the autocorrelation's shape and the PSD's shape are two views of one object, which is Wiener–Khinchin in Session 2. Ask which will have a low-pass spectrum before you get there.

**Pacing note.** The plotting line in this cell is dense (a conditional expression choosing `stem` or `plot`) — do not read it aloud. Project the figure and talk about the three traces.
</details>

## 2. Random Processes

💡 **Intuition.** A random process is a random variable *per time index* — one experiment produces a whole waveform (a *realization*). The process's personality lives in its joint statistics, but for signal processing two numbers usually suffice: the mean $\mu[n]$ and the **autocorrelation** $r[n, m] = E[x[n]x[m]]$ — how much the process remembers itself across time. **Wide-sense stationary (WSS)** means those two don't care about absolute time: $\mu$ constant, $r$ depends only on the lag $k = n - m$. Stationarity is what lets one long recording stand in for the whole ensemble (ergodicity — the [LLN](../Intro_Math/Analysis/Independence.ipynb) applied along time).

In [2]:
# Three processes, three memories: white, AR(1) smooth, AR(1) alternating
N, R = 2048, 400                       # length, realizations for ensemble averages
w = rng.standard_normal((R, N))
ar_pos = sig.lfilter([1], [1, -0.9], w, axis=1)    # remembers with + sign: smooth
ar_neg = sig.lfilter([1], [1, +0.9], w, axis=1)    # remembers with − sign: jittery

def acf_ensemble(X, maxlag=20):
    X = X - X.mean()
    return np.array([np.mean(X[:, :N-k] * X[:, k:]) for k in range(maxlag)])

fig, axes = plt.subplots(1, 2, figsize=(9.5, 2.8))
for name, X in [("white", w), ("AR(1) a=0.9", ar_pos), ("AR(1) a=−0.9", ar_neg)]:
    axes[0].plot(X[0, :200], alpha=0.7, label=name)
    r = acf_ensemble(X); axes[1].stem(np.arange(20), r / r[0], basefmt=" ", label=name) if name=="white" else axes[1].plot(r / r[0], "o-", alpha=0.7, label=name)
axes[0].set_title("one realization each"); axes[0].legend(fontsize=7)
axes[1].set_title("normalized autocorrelation r[k]/r[0]"); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

/tmp/ipykernel_2027473/1597755380.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Three processes, and the two panels show the same information twice. On the left, one realisation each: white noise looks like static, $a = 0.9$ looks smooth and wandering, $a = -0.9$ looks violently jittery. On the right, the autocorrelations: white noise is a spike at lag 0 and nothing after, $a = 0.9$ decays smoothly and positively, and $a = -0.9$ **alternates in sign** as it decays.

**The two AR processes are the instructive pair.** They have identical memory *strength* — $|a| = 0.9$ in both — and look nothing alike. The positive one remembers with a plus sign, so each sample resembles the last and the waveform is smooth. The negative one remembers with a minus sign, so each sample tends to be the opposite of the last, and the result looks like noise despite being just as strongly correlated. **Correlation strength and visual smoothness are different properties**, and the sign of the memory decides which you see. Anyone who equates "correlated" with "smooth" gets this wrong.

That also previews Session 2. Positive memory means energy concentrated at *low* frequency; alternating memory means energy at *high* frequency. The autocorrelation and the PSD are two views of the same object — which is precisely the Wiener–Khinchin theorem.

**Note the luxury this cell is quietly using.** `R = 400` realisations, and `acf_ensemble` averages across them — a genuine *ensemble* average, exactly as the definition $E[x[n]x[m]]$ demands. No real experiment gets 400 independent runs of the same process. In practice you have one recording and you replace the ensemble average with a **time** average, which is legal only under **ergodicity**: the assumption that one long realisation explores the same statistics the ensemble would. That is the [law of large numbers](../Intro_Math/Analysis/Independence.ipynb) applied along time instead of across trials, and every estimator in the rest of this workshop depends on it.

**And WSS is an assumption, not an observation.** Stationarity says the mean and autocorrelation do not depend on absolute time. These synthetic processes satisfy it by construction. Speech does not — its statistics change every phoneme. Nor does a radar return, or music, or an ECG. WSS is imposed to make the theory work, and much of the rest of the DSP track exists to handle what happens when it fails: [time–frequency analysis](./Time_Frequency_2.ipynb) for signals whose spectrum moves, [cyclostationarity](./Cyclostationary_HOS.ipynb) for signals whose statistics repeat periodically.

---
### 🕐 Session 2 of 4 — *The Power Spectral Density* (~40 min)
**Goal:** define the PSD; learn why the raw periodogram lies and how Welch fixes it.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (Wiener).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: The Power Spectral Density</b></summary>

**Timing (~40 min).** 8 min Wiener–Khinchin · 12 min why the periodogram is inconsistent · 12 min the demo · 8 min the resolution/variance trade.

**Board first — Wiener–Khinchin as a bridge, not a formula.** The PSD is the Fourier transform of the autocorrelation. Session 1's smooth AR process had slowly-decaying positive correlation, so its power sits at low frequency; the alternating one sits at high frequency. Two descriptions of one object. Once the room accepts that, the PSD stops being a new definition and becomes the frequency-domain view of memory.

**Then the counterintuitive result that is the point of the session.** The raw periodogram is **inconsistent**: its variance does *not* decrease as you collect more data. Students find this genuinely shocking, because every estimator they have met improves with $N$. Set it up as a paradox before resolving it.

**The mechanism, made concrete.** At each frequency, the DFT bin is essentially one complex Gaussian, so $|X(\omega)|^2$ is a $\chi^2_2$ variable — **two degrees of freedom, regardless of $N$**. Its standard deviation equals its mean, so the relative error is 100% forever. More data does not average anything at a fixed frequency; it just adds *more frequency bins*, each equally noisy. Getting students to see that "more data" and "more averaging" are not the same thing here is the real lesson.

**Then Welch as the obvious fix once the diagnosis is right.** If the problem is one degree of freedom per bin, get more: chop into $K$ segments, periodogram each, average. Now each bin is $\chi^2_{2K}$ and the relative error falls as $1/\sqrt{K}$. With $N = 32768$ and 512-sample segments at 50% overlap there are about 127 segments, so relative error drops from 100% to roughly **9%** — which is what makes the Welch trace on the plot lie on the true PSD instead of around it.

**Name the price explicitly.** Resolution. Full-length bins are $1/32768$ wide; 512-sample segments give bins 64× coarser. Ask what that costs — two spectral peaks closer than the segment bandwidth merge into one. So `nperseg` is a *variance-versus-resolution* dial, and the right setting depends on whether you are hunting narrow peaks or estimating a smooth spectrum. This is the same time–frequency trade as everywhere else in the track.

**Ask the room.** "The periodogram has 100% relative error. Why does the plot still look roughly right in shape?" Because the errors are independent across bins, so the *eye* averages them — the fuzz straddles the true curve. That is also why the periodogram remains fine for locating a strong narrow peak and useless for estimating a spectral *level*. Task determines estimator.

**If you have five minutes.** Re-run with `nperseg=64` and `nperseg=4096`: the first gives a very smooth but blurry estimate, the second is sharper and visibly noisier. Watching one knob move both properties in opposite directions is worth more than any explanation.
</details>

## 3. The PSD

💡 **Intuition.** The PSD is the autocorrelation's Fourier transform (Wiener–Khinchin): it says how the process's *power* is distributed over frequency — the ensemble version of the spectrum. The trap: the **raw periodogram** $|X(\omega)|^2/N$ is an *inconsistent* estimator — its variance never shrinks, no matter how much data you record, because each frequency bin is essentially one squared Gaussian (~1 degree of freedom, χ²₂-fluctuating forever). **Welch's fix**: chop into segments, periodogram each, *average* — trading resolution for the variance decay the [LLN](../Intro_Math/Analysis/Independence.ipynb) provides.

In [3]:
# One AR(2) process, its TRUE spectrum, and two estimates from the SAME data
b_ar, a_ar = [1.0], [1.0, -1.2, 0.81]
x = sig.lfilter(b_ar, a_ar, rng.standard_normal(2**15))

f_true = np.linspace(0, 0.5, 512)
_, H = sig.freqz(b_ar, a_ar, worN=2*np.pi*f_true)
psd_true = np.abs(H)**2

f_p, P_raw = sig.periodogram(x)
f_w, P_welch = sig.welch(x, nperseg=512)

plt.figure(figsize=(9, 3))
plt.semilogy(f_p, P_raw, alpha=0.35, label="raw periodogram (variance never dies)")
plt.semilogy(f_w, P_welch, linewidth=2, label="Welch, 512-sample segments")
plt.semilogy(f_true, psd_true, "k--", linewidth=1.5, label="true PSD")
plt.xlim(0, 0.5); plt.legend(); plt.xlabel("normalized frequency")
plt.title("32768 samples: the periodogram still fuzzes, Welch converges")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2027473/481309448.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Both estimates come from the **same 32768 samples**. The Welch curve sits on the true PSD; the raw periodogram fuzzes wildly around it, spanning more than an order of magnitude at neighbouring frequencies. More data did not fix the periodogram — and it never will.

**Why the periodogram is inconsistent, precisely.** At each frequency, the DFT coefficient is essentially one complex Gaussian, so $|X(\omega)|^2$ is a $\chi^2_2$ random variable: **two degrees of freedom, independent of $N$**. For $\chi^2_2$ the standard deviation equals the mean, so the relative error is **100%**, forever. Collecting more samples does not average anything at a given frequency; it merely produces *more frequency bins*, each just as noisy as before. This is the rare case where an estimator does not improve with data, and it is worth sitting with — "more data" and "more averaging" are not the same operation.

**Welch fixes the diagnosis rather than the symptom.** If the problem is one degree of freedom per bin, obtain more: chop into $K$ segments, periodogram each, average. Each bin becomes $\chi^2_{2K}$ and the relative error falls as $1/\sqrt{K}$. Here $N = 32768$ with 512-sample segments at 50% overlap gives roughly **127 segments**, so relative error drops from 100% to about **9%**. That factor of eleven is the entire visible difference between the two traces, and it is the [law of large numbers](../Intro_Math/Analysis/Independence.ipynb) doing exactly what it always does — once you arrange for something to actually be averaged.

**And the price is resolution.** Full-length bins would be $1/32768$ wide; 512-sample segments make them 64× coarser. Two spectral peaks closer together than the segment bandwidth will merge into one and no amount of averaging separates them. So `nperseg` is a **variance-versus-resolution dial**, not a performance setting: small segments give a smooth, blurry estimate; large segments give a sharp, noisy one. Try 64 and 4096 to see one knob move both properties in opposite directions.

**Which estimator is "right" depends on the question.** The periodogram remains perfectly serviceable for *locating* a strong narrow tone — a peak stands out even amid 100% fuzz — and is useless for *estimating a spectral level*, where you need the value rather than the position. Welch is the reverse. Notice too that the fuzz straddles the true curve rather than sitting off to one side: the periodogram is unbiased and merely high-variance, which is why the eye can still read its shape.

That distinction matters immediately: Session 3's Wiener filter is built entirely from PSD *values*, so it needs Welch. A periodogram-based Wiener filter would inherit 100% error in every gain it computes.

---
### 🕐 Session 3 of 4 — *The Wiener Filter, Derived* (~35 min)
**Goal:** solve the optimal linear filtering problem the adaptive filters approximate.
**Builds on:** Session 2; [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S2. &nbsp; **Feeds into:** Session 4 (detection).

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: The Wiener Filter, Derived</b></summary>

**Timing (~35 min).** 10 min the Wiener–Hopf derivation · 8 min the orthogonality principle · 10 min the frequency-domain form · 7 min the demo and its honest gain.

**This session pays a debt — say so.** The [adaptive filtering workshops](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) repeatedly referred to "the Wiener solution" as the target LMS chases without ever deriving it. Here it gets derived. Framing the session as settling an IOU gives it more weight than presenting it as a new topic.

**The derivation is three lines; do all three.** Minimise $E[(d - \mathbf{w}^\top\mathbf{x})^2]$, differentiate with respect to $\mathbf{w}$, set to zero, and the **Wiener–Hopf equations** $R\mathbf{w}_o = \mathbf{p}$ drop out. Nothing is hidden. Emphasise that this is an ordinary quadratic minimisation — the objective is convex in $\mathbf{w}$, so the stationary point is the global optimum, and there is nothing to iterate.

**The orthogonality principle is the geometric content — do not skip it.** $R\mathbf{w}_o = \mathbf{p}$ says exactly $E[e \cdot \mathbf{x}] = 0$: the optimal error is orthogonal to every observation. So the Wiener filter is a **projection** of $d$ onto the span of the observations, and that is the [Hilbert space](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) projection theorem in a new costume. Students who see the connection get a much better mental model than those who memorise the normal equations. Ask what "the error is orthogonal to the data" means intuitively — it means no further linear function of the data could reduce the error, i.e. you have extracted everything linearly extractable.

**The frequency-domain form is the memorable one.** $W(\omega) = S_d/(S_d + S_v)$: a per-frequency **trust dial**. Where the signal dominates, pass; where noise dominates, suppress. Optimal filtering is spectral triage. Point out the structural identity with the Kalman gain from [the Kalman workshop](../Intro_Time_Series/Intro_AdFilt_KF.ipynb) — signal variance over total variance, weighting by relative certainty. Same idea, one indexed by frequency and the other by time.

**Frame the 3.2 dB honestly.** Input SNR 10.1 dB, output 13.3 dB. That is a real gain and it is not dramatic, and students should understand why rather than assume the filter underperformed. The desired signal is AR(1) with $a = 0.95$, so its power is concentrated at low frequency but it is *not* band-limited — it overlaps the white noise everywhere. Wiener can only exploit the *difference* in spectral shape, and where signal and noise coexist in a band, the best possible linear filter still passes some noise and attenuates some signal. Ask what would raise the gain: a signal occupying a narrower band, or a coloured noise that avoids the signal band. Nothing about the filter changes — the achievable gain is a property of the *spectra*.

**Point at what the filter did and did not see.** `W` is built from `S_d` and the known noise level — PSDs only. It never touched `d`'s samples. The printed comment says "no peeking," and it is worth verifying aloud: this is second-order statistics doing all the work, which is exactly why Wiener filtering generalises to signals you have never seen.
</details>

## 4. Optimal Linear Filtering

Problem: estimate desired $d[n]$ from observations $x[n]$ using an FIR filter $\hat{d} = \mathbf{w}^T \mathbf{x}[n]$, minimizing $E[e^2]$.

Setting the gradient to zero (the [matrix calculus](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) you've done) gives the **Wiener–Hopf equations**
$$R \mathbf{w}_o = \mathbf{p}, \qquad R = E[\mathbf{x}\mathbf{x}^T], \;\; \mathbf{p} = E[d \, \mathbf{x}],$$
— a projection ([orthogonality principle](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb): the optimal error is orthogonal to every observation). [LMS/APA](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) chase this solution without knowing $R, \mathbf{p}$; here we *compute* it.

💡 **Intuition.** In the frequency domain the noncausal solution is transparent: $W(\omega) = \frac{S_d(\omega)}{S_d(\omega) + S_v(\omega)}$ for signal-plus-noise — a **per-frequency trust dial** (compare the Kalman gain!). Where signal dominates, pass ≈ 1; where noise dominates, squash ≈ 0. Optimal filtering is spectral triage.

In [4]:
# Wiener denoising, built from PSDs alone
d = sig.lfilter([1], [1, -0.95], rng.standard_normal(2**14))   # smooth desired signal
v = 1.0 * rng.standard_normal(2**14)                            # white noise
x = d + v

f, S_d = sig.welch(d, nperseg=1024)
_, S_x = sig.welch(x, nperseg=1024)
W = S_d / (S_d + 1.0)                                           # unit-variance white noise: S_v = 1

# apply as zero-phase frequency-domain filter (block processing)
X = np.fft.rfft(x)
fW = np.interp(np.fft.rfftfreq(len(x)), f, W)
dhat = np.fft.irfft(X * fW, n=len(x))

print(f"input SNR  {10*np.log10(np.var(d)/np.var(v)):5.1f} dB")
print(f"output SNR {10*np.log10(np.var(d)/np.var(dhat - d)):5.1f} dB   (Wiener gain, no peeking at d's samples — only its PSD)")

input SNR   10.1 dB
output SNR  13.3 dB   (Wiener gain, no peeking at d's samples — only its PSD)


**What just happened.** Input SNR **10.1 dB**, output SNR **13.3 dB** — a 3.2 dB gain, computed from **PSDs alone**. The filter never touched `d`'s samples; `W = S_d / (S_d + 1.0)` was built from a spectral estimate and a known noise level, and that was enough.

**The formula is a per-frequency trust dial.** $W(\omega) = S_d/(S_d + S_v)$ approaches 1 where the signal dominates and 0 where noise dominates, sliding smoothly between. Optimal filtering turns out to be *spectral triage*: at each frequency, weight by how much of what you are hearing is actually signal. Note the structural identity with the Kalman gain from [the Kalman workshop](../Intro_Time_Series/Intro_AdFilt_KF.ipynb) — signal variance over total variance — one indexed by frequency, the other by time. The same "trust each source in proportion to its certainty" logic runs through both.

**Now be honest about 3.2 dB, because it is a modest number and the reason matters.** The desired signal is AR(1) with $a = 0.95$, so its power concentrates at low frequency — but it is *not* band-limited. Its spectrum has tails that overlap the white noise across the entire band. Wiener can only exploit the *difference* in spectral shape, so wherever signal and noise genuinely coexist, even the optimal linear filter must pass some noise and attenuate some signal. **The achievable gain is a property of the two spectra, not of the filter.** Give the signal a narrower band, or give the noise a spectrum that avoids the signal band, and the same filter delivers far more. Nothing about the algorithm would change.

That is worth stating because it inverts the usual reading. A disappointing result here would not mean "use a better filter" — this *is* the best linear filter, provably. It means the problem itself has limited headroom, which is a much more useful diagnosis.

**Why the derivation matters as much as the result.** Setting $\partial E[e^2]/\partial\mathbf{w} = 0$ gives the Wiener–Hopf equations $R\mathbf{w}_o = \mathbf{p}$, and that condition says precisely $E[e\cdot\mathbf{x}] = 0$: **the optimal error is orthogonal to every observation**. So the Wiener filter is a *projection* of the desired signal onto the span of the data — the [Hilbert space projection theorem](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) appearing in yet another costume. Intuitively, orthogonality means no further linear function of the observations could reduce the error: everything linearly extractable has been extracted.

**And this closes a debt.** [LMS, NLMS, and APA](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) were all introduced as chasing "the Wiener solution" without one ever being computed, because $R$ and $\mathbf{p}$ are unknown in practice. Here we computed it. The adaptive filters are the online, statistics-free approximations to this cell; this cell is what they are approximating.

One caveat on the implementation: `W` is applied as a zero-phase frequency-domain multiply over the whole block, which is the *non-causal* Wiener filter. It uses future samples, so it is fine for offline processing and unavailable for real-time work. The causal version requires spectral factorisation and performs somewhat worse — a real distinction worth knowing before deploying this.

---
### 🕐 Session 4 of 4 — *Matched Filters & Detection* (~40 min)
**Goal:** detect a known pulse in noise optimally; meet ROC curves and the Neyman–Pearson view.
**Builds on:** Session 3; [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb).

---

<details>
<summary>🎓 <b>Teacher notes — Session 4: Matched Filters & Detection</b></summary>

**Timing (~40 min).** 8 min detection vs estimation · 10 min the matched filter · 10 min Neyman–Pearson and the ROC · 12 min the demo.

**Board first — the change of question.** Everything so far asked "what is the value?" Detection asks "**is it there at all?**" That is a hypothesis test, not an estimate, and it needs different machinery: two error types (false alarm and missed detection), a threshold, and a way to trade them. Students who have done [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) often try to force detection into an estimation frame; naming the difference up front prevents that.

**The matched filter, and why it is Cauchy–Schwarz.** For a *known* pulse in white Gaussian noise, the optimal detector correlates the data against the pulse. The reason is one line from [Hilbert Spaces](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb): $|\langle x, h\rangle| \le \|x\|\|h\|$ with equality exactly when $h \propto x$ — so among all linear statistics of fixed norm, correlating against the template maximises the output at the decision instant. Ask what "known pulse" is doing in that sentence; it is the entire assumption, and it is why radar (which knows what it transmitted) is the perfect application.

**Neyman–Pearson framed as a decision, not a formula.** You cannot minimise both error types at once. NP fixes the false-alarm rate at whatever you can tolerate and then maximises detection probability subject to it — and the likelihood-ratio test achieves that optimum. Ask the room to pick $P_{FA}$ for three scenarios: a radar looking for aircraft, a cancer screening test, a spam filter. The answers differ wildly, and that is the point — the threshold encodes a *cost judgement*, not a statistical fact.

**Teach reading the ROC.** Sweeping the threshold traces the curve; the diagonal is a coin flip; up-and-left is better; area under the curve summarises. The important structural point is that a curve *entirely above* another means dominance at **every** operating point — the better detector is better no matter what threshold you pick, so the comparison is threshold-free. That is why ROC is the universal report card across radar, medicine, and machine learning.

**Set up the comparison before running it.** The energy detector asks "is there more energy than usual?" The matched filter asks "is there energy *shaped like my pulse*?" Ask which should win and why. The energy detector discards the template entirely — it would respond identically to noise of the same power — so it throws away exactly the information that identifies the signal. This is the same lesson as [Compressed Sensing](./Compressed_Sensing.ipynb)'s L1-versus-L2: what you know about the signal is what buys performance.

**Be precise about when the matched filter is optimal.** Known pulse shape, known arrival time, white Gaussian noise. Relax any one and it degrades: unknown arrival time needs a bank of filters or a sliding correlation (which is what radar pulse compression does); coloured noise needs whitening first, then matching; unknown shape needs a generalised likelihood ratio test. The energy detector, for all its weakness here, needs *none* of that knowledge — which is exactly why [Cyclostationary detection](./Cyclostationary_HOS.ipynb) exists for signals whose structure you know only partially.
</details>

## 5. Detection

💡 **Intuition.** Estimation asks 'what is $\theta$?'; detection asks '**is it there at all?**' For a known pulse in white Gaussian noise, the optimal detector correlates the data against the pulse — the **matched filter**, which is just the [Cauchy–Schwarz](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) statement that correlation against a template is maximized by the template itself. It maximizes SNR at the decision instant; radar, sonar, GPS, and your Wi-Fi preamble sync all run on it.

**Neyman–Pearson framing.** Choose the threshold to fix the false-alarm rate $P_{FA}$; the likelihood-ratio test (here: matched filter output vs threshold) then maximizes detection probability $P_D$. Sweeping the threshold traces the **ROC curve** — the universal report card of any detector, from radar to medical tests to spam filters.

In [5]:
# Matched filter vs naive energy detector, at SNR where it matters
pulse = sig.gausspulse(np.linspace(-1, 1, 64), fc=4)
pulse /= np.linalg.norm(pulse)
trials, sigma = 4000, 1.2

def run(with_pulse):
    x = sigma * rng.standard_normal((trials, 64))
    if with_pulse: x += pulse
    mf = x @ pulse                      # matched filter statistic
    en = (x**2).sum(1)                  # energy detector statistic
    return mf, en

mf1, en1 = run(True); mf0, en0 = run(False)

def roc(stat1, stat0):
    ths = np.quantile(np.concatenate([stat0, stat1]), np.linspace(0, 1, 200))
    return [( (stat0 > t).mean(), (stat1 > t).mean()) for t in ths]

plt.figure(figsize=(4.6, 4.2))
for name, (s1, s0) in [("matched filter", (mf1, mf0)), ("energy detector", (en1, en0))]:
    pts = np.array(roc(s1, s0))
    plt.plot(pts[:, 0], pts[:, 1], label=name)
plt.plot([0, 1], [0, 1], "k:", linewidth=0.8, label="coin flip")
plt.xlabel("$P_{FA}$"); plt.ylabel("$P_D$"); plt.legend()
plt.title("ROC: the matched filter dominates at every threshold")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2027473/1927027460.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Two ROC curves from the same 4000 trials of the same data, and the matched filter's sits **entirely above** the energy detector's. That is a stronger statement than "it scores better": a curve that dominates everywhere means the matched filter wins at *every* operating point, whatever false-alarm rate you choose. The comparison is threshold-free, which is exactly what the ROC is for.

**Why the energy detector loses.** It computes $\sum x^2$ — "is there more energy than usual?" — and in doing so it *discards the template completely*. It would respond identically to noise of the same total power, or to a pulse of entirely the wrong shape. The matched filter computes $x \cdot h$, asking "is there energy **shaped like my pulse**?", and that extra question is worth the whole gap between the curves.

The general principle is one this curriculum keeps returning to: **what you know about the signal is what buys performance**. It is the same lesson as L1-versus-L2 in [Compressed Sensing](./Compressed_Sensing.ipynb) — the prior, not the data, is doing the work.

**And the matched filter is optimal for a provable reason.** Cauchy–Schwarz says $|\langle x, h\rangle| \le \|x\|\,\|h\|$ with equality precisely when $h \propto x$. So among all linear statistics of a given norm, correlating against the template maximises the output at the decision instant — this is the [Hilbert space](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) inequality doing detection theory. Not "a good idea that works well," but the best possible linear statistic, with a one-line proof.

**Read the ROC properly.** The diagonal is a coin flip — a detector on that line has no information. Up and to the left is better. Sweeping the threshold moves you *along* a curve, trading false alarms against missed detections, but never off it: the curve is the detector's capability, and the threshold is only where on it you choose to sit. That choice is a **cost judgement**, not a statistical one. A radar hunting aircraft, a cancer screening test, and a spam filter should sit at wildly different points on their curves, because the relative cost of a false alarm versus a miss differs in each. Neyman–Pearson formalises this: fix $P_{FA}$ at what you can tolerate, then maximise $P_D$ subject to it.

**The assumptions are narrow, and worth stating.** The matched filter is optimal for a *known* pulse, at a *known* arrival time, in *white Gaussian* noise. Relax any of the three and it degrades: unknown timing needs a sliding correlation or a filter bank — which is precisely what radar [pulse compression](./Radar_Signal_Processing.ipynb) is; coloured noise requires whitening first, then matching; an unknown pulse shape needs a generalised likelihood ratio test. The energy detector, for all its weakness here, needs *none* of that knowledge, which is why it survives in practice and why [cyclostationary detection](./Cyclostationary_HOS.ipynb) exists for the middle ground where you know the signal's structure but not its details.

## 6. Conclusion

WSS + ergodicity let one recording speak for the ensemble; Welch buys consistent spectra with the LLN; Wiener filtering is spectral triage and the target all adaptive filters chase; matched filtering + Neyman–Pearson is optimal 'is it there?'. This is the statistical spine of practical DSP.

---
## Where next

- [Adaptive Filtering](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) — Wiener pursued online.
- [Array Processing](./Array_Processing.ipynb) — these tools across space, not just time.
- [Digital Communications](./Digital_Communications.ipynb) — matched filters earning rent every symbol.